In [8]:
import pandas as pd
import os
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# AÑADIDO: .values.ravel() al final de las 'y' para evitar los errores rojos
X_train = pd.read_csv("C:\\Users\\alvarodela.herran\\Machine_learning\\Adv_ML\\Train_test_split\\X_train.csv")
y_train = pd.read_csv("C:\\Users\\alvarodela.herran\\Machine_learning\\Adv_ML\\Train_test_split\\y_train.csv").values.ravel()
X_test = pd.read_csv("C:\\Users\\alvarodela.herran\\Machine_learning\\Adv_ML\\Train_test_split\\X_test.csv")
y_test = pd.read_csv("C:\\Users\\alvarodela.herran\\Machine_learning\\Adv_ML\\Train_test_split\\y_test.csv").values.ravel()

target_folder = 'comparations'
os.makedirs(target_folder, exist_ok=True) # exist_ok=True prevents errors if the folder already exists

# 2. Updated function to save to the new path
def save_experiment(model_name, imbalance_method, accuracy, precision, recall, f1, roc_auc):
    new_result = {
        'Model': [model_name],
        'Imbalance_Method': [imbalance_method],
        'Accuracy': [round(accuracy, 4)],
        'Precision': [round(precision, 4)],
        'Recall': [round(recall, 4)],
        'F1_Score': [round(f1, 4)],
        'ROC_AUC': [round(roc_auc, 4)]
    }
    df_new = pd.DataFrame(new_result)
    
    # Point the file to inside the folder
    csv_file = f'{target_folder}/imbalance_results.csv'
    
    if os.path.exists(csv_file):
        df_new.to_csv(csv_file, mode='a', header=False, index=False)
    else:
        df_new.to_csv(csv_file, index=False)
        
    print(f"✅ Success! Results for {model_name} + {imbalance_method} saved in the '{target_folder}' folder.")

Voting classifier

In [12]:
# Importamos Pipeline de imblearn (especial para balanceadores)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("⏳ 1. Configurando los modelos base...")
clf_lr = LogisticRegression(random_state=42, max_iter=2000)
clf_rf = RandomForestClassifier(random_state=42, n_estimators=100)
clf_svc = SVC(probability=True, random_state=42)

# El Voting Classifier es nuestro "modelo final"
voting_clf = VotingClassifier(
    estimators=[('lr', clf_lr), ('rf', clf_rf), ('svc', clf_svc)], 
    voting='soft'
)

print("⏳ 2. Definiendo la lista de Pipelines...")
# Aquí creamos un Pipeline para cada técnica de la Unidad 3
experimentos = {
    'Ninguno (Baseline)': voting_clf, # Sin pipeline, solo el modelo
    'Random UnderSampler': ImbPipeline([('resample', RandomUnderSampler(random_state=42)), ('model', voting_clf)]),
    'SMOTE': ImbPipeline([('resample', SMOTE(random_state=42)), ('model', voting_clf)]),
    'ADASYN': ImbPipeline([('resample', ADASYN(random_state=42)), ('model', voting_clf)])
}



print("⏳ 3. ¡Iniciando el laboratorio automático con Pipelines!\n")

for nombre_metodo, pipe in experimentos.items():
    print(f"👉 Ejecutando: Voting Classifier + {nombre_metodo}...")
    
    # Entrenar el pipeline completo (primero balancea, luego entrena el voting)
    pipe.fit(X_train, y_train)
    
    # Predecir
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test) 
    
    # Calcular métricas (Aseguramos 'weighted' en todas para evitar el error multiclase)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    roc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
    
    # Guardar resultados
    save_experiment('Voting Classifier (LR+RF+SVC)', nombre_metodo, acc, prec, rec, f1, roc)

print("\n🎉 ¡Todo listo! Ya tienes las 4 filas en tu CSV de la carpeta 'comparations'.")

⏳ 1. Configurando los modelos base...
⏳ 2. Definiendo la lista de Pipelines...
⏳ 3. ¡Iniciando el laboratorio automático con Pipelines!

👉 Ejecutando: Voting Classifier + Ninguno (Baseline)...
✅ Success! Results for Voting Classifier (LR+RF+SVC) + Ninguno (Baseline) saved in the 'comparations' folder.
👉 Ejecutando: Voting Classifier + Random UnderSampler...
✅ Success! Results for Voting Classifier (LR+RF+SVC) + Random UnderSampler saved in the 'comparations' folder.
👉 Ejecutando: Voting Classifier + SMOTE...


AttributeError: 'NoneType' object has no attribute 'split'